# Minimal sanity experiments for text-support conflict

这个 notebook 用来做论文前的最小可行验证。它不实现完整 Dirichlet / evidential fusion，而是先验证 4 个 sanity experiments 是否过关。

四个实验：

1. **Text-support conflict 是否真的预测错误？**
2. **support label noise 下，conflict 是否比 entropy 更敏感？**
3. **prompt mismatch 下，能否识别 text prior 不可靠？**
4. **OOD / near-OOD query 下，ignorance 与 conflict 是否可分离？**

核心思想：

- text source: CLIP zero-shot prior
- support source: support cache / kNN evidence
- conflict: JS divergence、top-1 disagreement、margin conflict
- ignorance: support density low
- dominance: one source confidence clearly dominates another source

默认只跑轻量配置。实验 4 的 ImageNet-R/A/Sketch 可能较重，默认关闭，需要手动打开。


In [ ]:
from __future__ import annotations

import os
import sys
import json
import math
import random
import importlib
from pathlib import Path
from types import SimpleNamespace
from dataclasses import dataclass

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 200)

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


## 0. Configuration

把这个文件放在 `MMRL/notebooks/` 下运行。默认 `PROJECT_ROOT = ..`，即 `MMRL` 目录。

建议第一轮：

- `DATASET = "caltech101"` 或 `"dtd"` / `"eurosat"` / `"ucf101"`
- `SHOTS = 1, 2, 4, 16`
- `METHOD_ALIAS = "CLAP"` 或 `"TipA"` / `"BayesAdapter"` / `"PP_PROKER_ONEHOT"`

注意：这个 notebook 的 support cache prediction 是独立重建的 kNN/cache evidence，不要求你已经训练 Tip-Adapter；但如果你用已有 adapter checkpoint，也会加载它以保持项目 classnames / transform / CLIP backbone 一致。


In [ ]:
# ---- paths ----
PROJECT_ROOT = Path("..").resolve()
assert (PROJECT_ROOT / "run.py").exists(), f"PROJECT_ROOT seems wrong: {PROJECT_ROOT}"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = "DATASETS"
OUTPUT_ROOT = PROJECT_ROOT / "output_refactor"

# ---- base experiment config ----
METHOD_ALIAS = "CLAP"       # examples: "CLAP", "TipA", "BayesAdapter", "PP_PROKER_ONEHOT"
DATASET = "caltech101"
SHOTS = 16
SEED = 1
PROTOCOL = "FS"
EXEC_MODE = "cache"
BACKBONE = "ViT-B/16"

# Existing trained adapter directory. Used to load the trained adapter if available.
# For the sanity tests, zero-shot text and support-cache evidence are reconstructed from the built model/data.
RUN_TAG_BY_ALIAS = {
    "ZS": "ZS",
    "CLAP": "CLAP",
    "PP_PROKER_ONEHOT": "PP_PROKER_ONEHOT",
    "ECKA": "ECKA",
    "CAPEL": "CAPEL",
    "VNC_CAPEL": "VNC_CAPEL",
    "BayesAdapter": "BAYES_ADAPTER",
    "BAYES_ADAPTER": "BAYES_ADAPTER",
    "TipA": "TipA",
    "TipA-f-": "TipA-f-",
    "CrossModal": "CrossModal",
}
RUN_TAG = RUN_TAG_BY_ALIAS.get(METHOD_ALIAS, METHOD_ALIAS)
BACKBONE_DIR = BACKBONE.replace("/", "-")

MODEL_DIR = OUTPUT_ROOT / "ClipAdapters" / RUN_TAG / PROTOCOL / "fewshot_train" / DATASET / f"shots_{SHOTS}" / BACKBONE_DIR / f"seed{SEED}"
LOAD_EPOCH = None

OUT_DIR = OUTPUT_ROOT / "analysis" / "minimal_sanity_experiments" / RUN_TAG / DATASET / f"shots_{SHOTS}" / BACKBONE_DIR / f"seed{SEED}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- cache / fusion knobs ----
CACHE_BETA = 20.0          # support affinity temperature
CACHE_TOPK = 64            # top-k support neighbors used for density statistics
FINAL_BLEND = 0.5          # p_final = (1-FINAL_BLEND)*p_text + FINAL_BLEND*p_support
CONFLICT_GAMMA = 5.0       # confidence penalty for conflict-aware selective risk proxy
DENSITY_LOW_Q = 0.10
CONFLICT_HIGH_Q = 0.90
DOMINANCE_GAP = 0.30

# ---- which experiments to run ----
RUN_EXP1 = True
RUN_EXP2 = True
RUN_EXP3 = True
RUN_EXP4 = False  # heavy; turn on after ImageNet support/query datasets are ready

LABEL_NOISE_RATES = [0.0, 0.10, 0.20, 0.40]

PROMPT_MISMATCH_MODES = [
    "original",
    "weak_blend_mean",
    "coarse_alphabetic_groups",
    "wrong_permutation",
]

# OOD / near-OOD experiment. Recommended after quick ID sanity succeeds.
EXP4_SUPPORT_DATASET = "imagenet"
EXP4_QUERY_DATASETS = ["imagenetv2", "imagenet_r", "imagenet_a", "imagenet_sketch"]
EXP4_SUPPORT_SHOTS = 16

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR:", MODEL_DIR, "exists:", MODEL_DIR.exists())
print("OUT_DIR:", OUT_DIR)


## 1. Project integration helpers

这些 helper 复用项目的 `run.py` 方式构造 trainer / dataloader / CLIP model。  
如果 checkpoint 不存在，也可以设置 `LOAD_CHECKPOINT=False`，只用当前 config 构造 zero-shot 和 support cache sanity experiments。


In [ ]:
METHOD_CONFIG_BY_ALIAS = {
    "ZS": "configs/methods/clip_adapters_zs.yaml",
    "CLAP": "configs/methods/clip_adapters_clap.yaml",
    "PP_PROKER_ONEHOT": "configs/methods/clip_adapters_pp_proker_onehot.yaml",
    "ECKA": "configs/methods/clip_adapters_ecka.yaml",
    "CAPEL": "configs/methods/clip_adapters_capel.yaml",
    "VNC_CAPEL": "configs/methods/clip_adapters_vnccapel.yaml",
    "RANDOM": "configs/methods/clip_adapters_random.yaml",
    "TR": "configs/methods/clip_adapters_tr.yaml",
    "TaskRes": "configs/methods/clip_adapters_tr.yaml",
    "ClipA": "configs/methods/clip_adapters_clipa.yaml",
    "CLIPA": "configs/methods/clip_adapters_clipa.yaml",
    "TipA": "configs/methods/clip_adapters_tipa.yaml",
    "TipA-f-": "configs/methods/clip_adapters_tipa_f.yaml",
    "TipA-F": "configs/methods/clip_adapters_tipa_f.yaml",
    "CrossModal": "configs/methods/clip_adapters_crossmodal.yaml",
    "BayesAdapter": "configs/methods/clip_adapters_bayes.yaml",
    "BAYES_ADAPTER": "configs/methods/clip_adapters_bayes.yaml",
    "BayesAdapter_l2": "configs/methods/clip_adapters_bayes_clap.yaml",
    "BAYES_ADAPTER_l2": "configs/methods/clip_adapters_bayes_clap.yaml",
}

def method_config_for_alias(alias: str) -> str:
    if alias not in METHOD_CONFIG_BY_ALIAS:
        raise KeyError(f"Unknown METHOD_ALIAS={alias}. Known: {sorted(METHOD_CONFIG_BY_ALIAS)}")
    return METHOD_CONFIG_BY_ALIAS[alias]

def dataset_config(dataset: str) -> str:
    return f"configs/datasets/{dataset}.yaml"

def protocol_config(protocol: str) -> str:
    if protocol == "FS":
        return "configs/protocols/fs.yaml"
    if protocol == "B2N":
        return "configs/protocols/b2n.yaml"
    if protocol == "CD":
        return "configs/protocols/cd.yaml"
    raise ValueError(f"Unknown protocol: {protocol}")

def build_output_dir(method_alias: str, dataset: str, shots: int, seed: int, backbone: str = BACKBONE, output_root: Path = OUTPUT_ROOT) -> Path:
    run_tag = RUN_TAG_BY_ALIAS.get(method_alias, method_alias)
    return output_root / "ClipAdapters" / run_tag / "FS" / "fewshot_train" / dataset / f"shots_{shots}" / backbone.replace("/", "-") / f"seed{seed}"


In [ ]:
from dassl.engine import build_trainer
from dassl.utils import set_random_seed, setup_logger
from core.config import setup_cfg
from core.utils import import_optional_modules

def register_project_modules():
    import_optional_modules([
        "datasets.oxford_pets", "datasets.oxford_flowers", "datasets.fgvc_aircraft",
        "datasets.dtd", "datasets.eurosat", "datasets.stanford_cars", "datasets.food101",
        "datasets.sun397", "datasets.caltech101", "datasets.ucf101", "datasets.imagenet",
        "datasets.imagenetv2", "datasets.imagenet_sketch", "datasets.imagenet_a", "datasets.imagenet_r",
    ])
    importlib.import_module("trainers.refactor_runner")

def make_args(
    method_alias: str,
    dataset: str,
    shots: int,
    seed: int,
    output_dir: Path,
    protocol: str = "FS",
    exec_mode: str = "cache",
    backbone: str = "ViT-B/16",
):
    mcfg = method_config_for_alias(method_alias)
    dcfg = dataset_config(dataset)
    pcfg = protocol_config(protocol)
    rcfg = "configs/runtime/adapter_family.yaml"

    for rel in [mcfg, dcfg, pcfg, rcfg]:
        assert (PROJECT_ROOT / rel).exists(), f"Missing config: {PROJECT_ROOT / rel}"

    return SimpleNamespace(
        root=str(PROJECT_ROOT / DATA_ROOT),
        output_dir=str(output_dir),
        dataset_config_file=str(PROJECT_ROOT / dcfg),
        method_config_file=str(PROJECT_ROOT / mcfg),
        protocol_config_file=str(PROJECT_ROOT / pcfg),
        runtime_config_file=str(PROJECT_ROOT / rcfg),
        exp_config="",
        method="ClipAdapters",
        protocol=protocol,
        exec_mode=exec_mode,
        seed=seed,
        trainer="RefactorRunner",
        eval_only=True,
        model_dir="",
        load_epoch=None,
        no_train=True,
        opts=[
            "DATASET.NUM_SHOTS", str(shots),
            "DATASET.SUBSAMPLE_CLASSES", "all",
            "MODEL.BACKBONE.NAME", backbone,
        ],
    )

def build_sanity_trainer(
    method_alias: str = METHOD_ALIAS,
    dataset: str = DATASET,
    shots: int = SHOTS,
    seed: int = SEED,
    output_dir: Path = OUT_DIR,
    load_checkpoint: bool = True,
    model_dir: Path | None = None,
):
    register_project_modules()
    if seed >= 0:
        set_random_seed(seed)

    args = make_args(method_alias, dataset, shots, seed, output_dir)
    cfg = setup_cfg(args)
    setup_logger(cfg.OUTPUT_DIR)

    trainer = build_trainer(cfg)
    if load_checkpoint:
        if model_dir is None:
            model_dir = build_output_dir(method_alias, dataset, shots, seed)
        if Path(model_dir).exists():
            trainer.load_model(str(model_dir), epoch=LOAD_EPOCH)
            print(f"Loaded checkpoint from {model_dir}")
        else:
            print(f"[WARN] Checkpoint dir not found, using freshly built model: {model_dir}")
    trainer.set_model_mode("eval")
    return trainer

trainer = build_sanity_trainer(
    method_alias=METHOD_ALIAS,
    dataset=DATASET,
    shots=SHOTS,
    seed=SEED,
    output_dir=OUT_DIR,
    load_checkpoint=True,
    model_dir=MODEL_DIR,
)

method = trainer.method
model = trainer.method.model
device = trainer.device
classnames = list(trainer.dm.dataset.classnames)

print("method:", getattr(method, "method_name", None))
print("adapter:", type(model.adapter).__name__, getattr(model.adapter, "initialization_name", None))
print("device:", device)
print("num classes:", len(classnames))
print("train support samples:", len(trainer.dm.dataset.train_x))
print("test samples:", len(trainer.test_loader.dataset))


## 2. Core math helpers

统一约定：

- 高 `error_score` 表示更可能错。
- AURC 里低 score 的样本先保留，高 score 的样本先拒绝。


In [ ]:
EPS = 1e-12

def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    z = logits - np.max(logits, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.clip(e.sum(axis=1, keepdims=True), EPS, None)

def normalize_probs(p: np.ndarray) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=np.float64), EPS, None)
    return p / np.clip(p.sum(axis=1, keepdims=True), EPS, None)

def entropy_np(p: np.ndarray) -> np.ndarray:
    p = normalize_probs(p)
    return -(p * np.log(p)).sum(axis=1)

def js_divergence_np(p: np.ndarray, q: np.ndarray) -> np.ndarray:
    p = normalize_probs(p)
    q = normalize_probs(q)
    m = 0.5 * (p + q)
    kl_pm = (p * (np.log(p) - np.log(m))).sum(axis=1)
    kl_qm = (q * (np.log(q) - np.log(m))).sum(axis=1)
    return 0.5 * (kl_pm + kl_qm)

def cosine_distance_np(p: np.ndarray, q: np.ndarray) -> np.ndarray:
    p = normalize_probs(p)
    q = normalize_probs(q)
    num = (p * q).sum(axis=1)
    den = np.linalg.norm(p, axis=1) * np.linalg.norm(q, axis=1)
    return 1.0 - num / np.clip(den, EPS, None)

def margin_np(p: np.ndarray) -> np.ndarray:
    p = normalize_probs(p)
    top2 = np.partition(p, kth=-2, axis=1)[:, -2:]
    top2.sort(axis=1)
    return top2[:, 1] - top2[:, 0]

def energy_from_logits_np(logits: np.ndarray) -> np.ndarray:
    # Higher means less confident / more error-like.
    x = torch.tensor(logits).float()
    return (-torch.logsumexp(x, dim=1)).numpy()

def safe_roc_auc(y_binary: np.ndarray, score_high_for_positive: np.ndarray) -> float:
    y = np.asarray(y_binary).astype(int)
    s = np.asarray(score_high_for_positive, dtype=float)
    mask = np.isfinite(s)
    if mask.sum() < 2 or len(np.unique(y[mask])) < 2:
        return float("nan")
    return float(roc_auc_score(y[mask], s[mask]))

def aurc_from_error_score(error: np.ndarray, score_high_means_error: np.ndarray) -> float:
    error = np.asarray(error).astype(float)
    score = np.asarray(score_high_means_error, dtype=float)
    mask = np.isfinite(score)
    error = error[mask]
    score = score[mask]
    if len(error) == 0:
        return float("nan")
    order = np.argsort(score)
    e = error[order]
    risks = np.cumsum(e) / np.arange(1, len(e) + 1)
    return float(np.mean(risks))

def optimal_aurc(error: np.ndarray) -> float:
    e = np.asarray(error).astype(float)
    if len(e) == 0:
        return float("nan")
    order = np.argsort(e)
    e = e[order]
    risks = np.cumsum(e) / np.arange(1, len(e) + 1)
    return float(np.mean(risks))

def detection_table(df: pd.DataFrame, error_col: str, score_cols: list[str]) -> pd.DataFrame:
    rows = []
    error = df[error_col].to_numpy().astype(int)
    opt = optimal_aurc(error)
    for col in score_cols:
        score = df[col].to_numpy(dtype=float)
        auroc = safe_roc_auc(error, score)
        aurc = aurc_from_error_score(error, score)
        rows.append({
            "score": col,
            "auroc": auroc,
            "aurc": aurc,
            "eaurc": aurc - opt if np.isfinite(aurc) and np.isfinite(opt) else np.nan,
        })
    return pd.DataFrame(rows).sort_values(["auroc", "eaurc"], ascending=[False, True])

def topk_str(p: np.ndarray, names: list[str], k: int = 3) -> list[str]:
    out = []
    for row in normalize_probs(p):
        idx = np.argsort(-row)[:k]
        out.append("; ".join(f"{names[j]}:{row[j]:.4f}" for j in idx))
    return out


## 3. Feature and prediction collectors

这里重建：

- `p_text`: CLIP zero-shot prediction
- `p_support`: support cache / kNN prediction
- `p_final`: simple mixture prediction，用于 sanity test 的最终 prediction

support cache prediction 的定义：

```python
affinity = softmax(beta * cos(query_feature, support_feature))
p_support = affinity @ one_hot(support_label)
```

这不是完整 Tip-Adapter 训练，只是最小可行的 support evidence。


In [ ]:
@torch.no_grad()
def collect_query_features(trainer, data_loader=None, max_batches: int | None = None):
    trainer.set_model_mode("eval")
    model = trainer.method.model
    device = trainer.device
    if data_loader is None:
        data_loader = trainer.test_loader

    features_all = []
    labels_all = []
    paths = []

    for batch_idx, batch in enumerate(data_loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        image = batch["img"].to(device)
        label = batch["label"].to(device).long()

        try:
            feats = model.image_encoder(image.type(model.dtype))
        except Exception:
            feats = model.image_encoder(image.float())

        features_all.append(feats.detach().cpu().float())
        labels_all.append(label.detach().cpu().long())

        p = batch.get("impath", None) or batch.get("path", None)
        if p is None:
            p = [None] * int(label.shape[0])
        if isinstance(p, (list, tuple)):
            paths.extend(list(p))
        else:
            paths.extend([None] * int(label.shape[0]))

    return {
        "features": torch.cat(features_all, dim=0),
        "labels": torch.cat(labels_all, dim=0),
        "paths": paths,
    }

@torch.no_grad()
def get_support_features_from_trainer(trainer):
    # In cache exec mode, CacheExecutor already builds features_train / labels_train.
    if hasattr(trainer, "features_train") and hasattr(trainer, "labels_train"):
        return {
            "features": trainer.features_train.detach().cpu().float(),
            "labels": trainer.labels_train.detach().cpu().long(),
        }

    # Fallback: extract train loader features online.
    trainer.set_model_mode("eval")
    model = trainer.method.model
    device = trainer.device
    features_all = []
    labels_all = []
    for batch in trainer.train_loader_x:
        image = batch["img"].to(device)
        label = batch["label"].to(device).long()
        try:
            feats = model.image_encoder(image.type(model.dtype))
        except Exception:
            feats = model.image_encoder(image.float())
        features_all.append(feats.detach().cpu().float())
        labels_all.append(label.detach().cpu().long())
    return {
        "features": torch.cat(features_all, dim=0),
        "labels": torch.cat(labels_all, dim=0),
    }

def zero_shot_logits_from_features(trainer, query_features: torch.Tensor, text_features_override: torch.Tensor | None = None, batch_size: int = 2048) -> np.ndarray:
    model = trainer.method.model
    device = trainer.device
    if text_features_override is None:
        text_features = model.adapter.base_text_features.detach().float()
    else:
        text_features = text_features_override.detach().float()

    text_features = F.normalize(text_features.to(device), dim=-1)
    logit_scale = model.logit_scale.exp().detach().to(device)

    outs = []
    for start in range(0, query_features.shape[0], batch_size):
        q = query_features[start:start + batch_size].to(device).float()
        q = F.normalize(q, dim=-1)
        logits = q @ text_features.t() * logit_scale
        outs.append(logits.detach().cpu())
    return torch.cat(outs, dim=0).numpy()

def support_cache_probs(
    query_features: torch.Tensor,
    support_features: torch.Tensor,
    support_labels: torch.Tensor,
    num_classes: int,
    beta: float = CACHE_BETA,
    topk: int = CACHE_TOPK,
    batch_size: int = 1024,
):
    q_all = F.normalize(query_features.float(), dim=-1)
    s_all = F.normalize(support_features.float(), dim=-1)
    y = support_labels.long().cpu()

    onehot = F.one_hot(y, num_classes=num_classes).float()
    probs_all = []
    support_logits_all = []
    density_rows = []

    for start in range(0, q_all.shape[0], batch_size):
        q = q_all[start:start + batch_size]
        sim = q @ s_all.t()
        aff_logits = beta * sim
        aff = torch.softmax(aff_logits, dim=1)
        probs = aff @ onehot
        probs = probs / probs.sum(dim=1, keepdim=True).clamp_min(EPS)

        # A class-level support logit proxy: log cache probability.
        support_logits = torch.log(probs.clamp_min(EPS))

        k = min(int(topk), sim.shape[1])
        topv = torch.topk(sim, k=k, dim=1).values
        density_rows.append(torch.stack([
            sim.max(dim=1).values,
            topv.mean(dim=1),
            topv[:, :min(5, k)].mean(dim=1),
        ], dim=1))

        probs_all.append(probs.cpu())
        support_logits_all.append(support_logits.cpu())

    return {
        "probs": torch.cat(probs_all, dim=0).numpy(),
        "logits": torch.cat(support_logits_all, dim=0).numpy(),
        "density": torch.cat(density_rows, dim=0).numpy(),
    }

def blend_probs(p_text: np.ndarray, p_support: np.ndarray, blend: float = FINAL_BLEND) -> np.ndarray:
    p = (1.0 - blend) * normalize_probs(p_text) + blend * normalize_probs(p_support)
    return normalize_probs(p)

def margin_conflict_score(p_text: np.ndarray, p_support: np.ndarray) -> np.ndarray:
    pt = normalize_probs(p_text)
    ps = normalize_probs(p_support)
    top_t = pt.argmax(axis=1)
    top_s = ps.argmax(axis=1)
    rows = np.arange(pt.shape[0])
    mt = pt[rows, top_t] - pt[rows, top_s]
    ms = ps[rows, top_s] - ps[rows, top_t]
    disagree = (top_t != top_s).astype(float)
    return disagree * (np.maximum(mt, 0.0) + np.maximum(ms, 0.0))


In [ ]:
def build_text_support_frame(
    trainer,
    support_features: torch.Tensor | None = None,
    support_labels: torch.Tensor | None = None,
    text_features_override: torch.Tensor | None = None,
    query_pack: dict | None = None,
    beta: float = CACHE_BETA,
    final_blend: float = FINAL_BLEND,
    save_prefix: str | None = None,
):
    names = list(trainer.dm.dataset.classnames)
    num_classes = len(names)

    if support_features is None or support_labels is None:
        support = get_support_features_from_trainer(trainer)
        support_features = support["features"]
        support_labels = support["labels"]

    if query_pack is None:
        query_pack = collect_query_features(trainer)

    q_features = query_pack["features"]
    labels = query_pack["labels"].numpy().astype(int)
    paths = query_pack.get("paths", [None] * len(labels))

    text_logits = zero_shot_logits_from_features(trainer, q_features, text_features_override=text_features_override)
    p_text = softmax_np(text_logits)

    support_pred = support_cache_probs(
        query_features=q_features,
        support_features=support_features,
        support_labels=support_labels,
        num_classes=num_classes,
        beta=beta,
    )
    p_support = support_pred["probs"]
    support_logits = support_pred["logits"]
    density = support_pred["density"]

    p_final = blend_probs(p_text, p_support, blend=final_blend)
    final_logits = np.log(np.clip(p_final, EPS, 1.0))

    pred_text = p_text.argmax(axis=1)
    pred_support = p_support.argmax(axis=1)
    pred_final = p_final.argmax(axis=1)

    df = pd.DataFrame({
        "sample_index": np.arange(len(labels)),
        "path": paths,
        "y": labels,
        "y_name": [names[y] if 0 <= y < len(names) else str(y) for y in labels],
        "pred_text": pred_text,
        "pred_support": pred_support,
        "pred_final": pred_final,
        "pred_text_name": [names[i] for i in pred_text],
        "pred_support_name": [names[i] for i in pred_support],
        "pred_final_name": [names[i] for i in pred_final],
        "correct_text": (pred_text == labels).astype(int),
        "correct_support": (pred_support == labels).astype(int),
        "correct_final": (pred_final == labels).astype(int),
    })
    df["error_text"] = 1 - df["correct_text"]
    df["error_support"] = 1 - df["correct_support"]
    df["error_final"] = 1 - df["correct_final"]

    # conflict
    df["js_text_support"] = js_divergence_np(p_text, p_support)
    df["one_minus_cos_text_support"] = cosine_distance_np(p_text, p_support)
    df["top1_disagree_text_support"] = (pred_text != pred_support).astype(int)
    df["margin_conflict_text_support"] = margin_conflict_score(p_text, p_support)

    # text confidence
    df["msp_text"] = p_text.max(axis=1)
    df["entropy_text"] = entropy_np(p_text)
    df["margin_text"] = margin_np(p_text)
    df["energy_text"] = energy_from_logits_np(text_logits)

    # support confidence
    df["msp_support"] = p_support.max(axis=1)
    df["entropy_support"] = entropy_np(p_support)
    df["margin_support"] = margin_np(p_support)
    df["energy_support"] = energy_from_logits_np(support_logits)

    # final confidence
    df["msp_final"] = p_final.max(axis=1)
    df["entropy_final"] = entropy_np(p_final)
    df["margin_final"] = margin_np(p_final)
    df["energy_final"] = energy_from_logits_np(final_logits)

    # larger = more error-like
    for src in ["text", "support", "final"]:
        df[f"one_minus_msp_{src}"] = 1.0 - df[f"msp_{src}"]
        df[f"neg_margin_{src}"] = -df[f"margin_{src}"]

    # density / ignorance
    df["support_density_maxsim"] = density[:, 0]
    df["support_density_topk_mean"] = density[:, 1]
    df["support_density_top5_mean"] = density[:, 2]
    df["ignorance_score"] = -df["support_density_topk_mean"]

    # conflict-aware confidence proxy, not a real evidential fusion method.
    df["conflict_aware_confidence"] = df["msp_final"] / (1.0 + CONFLICT_GAMMA * df["js_text_support"])
    df["conflict_aware_error_score"] = 1.0 - df["conflict_aware_confidence"]

    df["top3_text"] = topk_str(p_text, names, k=3)
    df["top3_support"] = topk_str(p_support, names, k=3)
    df["top3_final"] = topk_str(p_final, names, k=3)

    pack = {
        "df": df,
        "p_text": p_text,
        "p_support": p_support,
        "p_final": p_final,
        "text_logits": text_logits,
        "support_logits": support_logits,
        "final_logits": final_logits,
        "labels": labels,
        "classnames": names,
    }

    if save_prefix is not None:
        csv_path = OUT_DIR / f"{save_prefix}_samples.csv"
        npz_path = OUT_DIR / f"{save_prefix}_probs.npz"
        df.to_csv(csv_path, index=False)
        np.savez_compressed(
            npz_path,
            p_text=p_text,
            p_support=p_support,
            p_final=p_final,
            text_logits=text_logits,
            support_logits=support_logits,
            labels=labels,
            classnames=np.array(names, dtype=object),
        )
        print("saved:", csv_path)
        print("saved:", npz_path)

    return pack


## Experiment 1: Text-support conflict 是否真的预测错误？

问题：

> conflict score 对 final prediction error 的 AUROC 是否明显高于 MSP / entropy / energy？

如果 `JS / top-1 disagreement / margin conflict` 明显不如 `MSP / entropy / energy`，则方向不成立或至少不是这个切口。


In [ ]:
if RUN_EXP1:
    support = get_support_features_from_trainer(trainer)
    query_pack = collect_query_features(trainer)
    exp1 = build_text_support_frame(
        trainer,
        support_features=support["features"],
        support_labels=support["labels"],
        query_pack=query_pack,
        save_prefix="exp1_text_support_conflict",
    )
    exp1_df = exp1["df"]

    exp1_scores = [
        "js_text_support",
        "one_minus_cos_text_support",
        "top1_disagree_text_support",
        "margin_conflict_text_support",
        "one_minus_msp_final",
        "entropy_final",
        "neg_margin_final",
        "energy_final",
        "one_minus_msp_text",
        "entropy_text",
        "one_minus_msp_support",
        "entropy_support",
        "conflict_aware_error_score",
    ]

    exp1_metrics = detection_table(exp1_df, "error_final", exp1_scores)
    exp1_metrics_path = OUT_DIR / "exp1_error_detection_metrics.csv"
    exp1_metrics.to_csv(exp1_metrics_path, index=False)

    print("final accuracy:", exp1_df["correct_final"].mean())
    print("text accuracy:", exp1_df["correct_text"].mean())
    print("support accuracy:", exp1_df["correct_support"].mean())
    print("text-support disagreement:", exp1_df["top1_disagree_text_support"].mean())
    print("saved:", exp1_metrics_path)
    display(exp1_metrics)
else:
    print("RUN_EXP1=False")


In [ ]:
if RUN_EXP1:
    # Main kill-test for Exp1: does JS add value after confidence?
    FEATURES_BASE = ["entropy_final", "margin_final", "msp_final"]
    FEATURES_PLUS_JS = FEATURES_BASE + ["js_text_support"]

    def heldout_logreg(df, features, error_col="error_final", seed=SEED, test_size=0.3):
        sub = df[[error_col] + features].replace([np.inf, -np.inf], np.nan).dropna()
        y = sub[error_col].astype(int).to_numpy()
        X = sub[features].to_numpy(dtype=float)
        if len(np.unique(y)) < 2:
            return {"features": features, "auroc": np.nan, "aurc": np.nan, "eaurc": np.nan}
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
        clf.fit(Xtr, ytr)
        prob = clf.predict_proba(Xte)[:, 1]
        aurc = aurc_from_error_score(yte, prob)
        return {
            "features": features,
            "n_train": len(ytr),
            "n_test": len(yte),
            "auroc": safe_roc_auc(yte, prob),
            "aurc": aurc,
            "eaurc": aurc - optimal_aurc(yte),
            "coef": dict(zip(features, clf.named_steps["logisticregression"].coef_[0])),
        }

    r_base = heldout_logreg(exp1_df, FEATURES_BASE)
    r_js = heldout_logreg(exp1_df, FEATURES_PLUS_JS)

    exp1_kill = pd.DataFrame([
        {k: v for k, v in r_base.items() if k != "coef"} | {"model": "confidence_only"},
        {k: v for k, v in r_js.items() if k != "coef"} | {"model": "confidence_plus_js"},
    ])
    exp1_kill["delta_auroc_vs_base"] = exp1_kill["auroc"] - float(exp1_kill.loc[exp1_kill.model == "confidence_only", "auroc"].iloc[0])
    exp1_kill["delta_eaurc_vs_base"] = exp1_kill["eaurc"] - float(exp1_kill.loc[exp1_kill.model == "confidence_only", "eaurc"].iloc[0])

    exp1_kill_path = OUT_DIR / "exp1_kill_test_confidence_plus_js.csv"
    exp1_kill.to_csv(exp1_kill_path, index=False)
    print("JS coef in plus model:", r_js.get("coef", {}).get("js_text_support"))
    print("saved:", exp1_kill_path)
    display(exp1_kill)


## Experiment 2: support label noise 下，conflict 是否比 entropy 更敏感？

做法：

- 对 support labels 注入 10%、20%、40% label noise。
- query set 不变。
- 重算 `p_support`, `p_final`, conflict。
- 比较 conflict vs entropy / MSP 对 error detection 的 AUROC / AURC。
- 额外比较 naive confidence selective risk 和 conflict-aware confidence proxy。

重点看：

- noise 越高，support 是否过度自信？
- conflict AUROC 是否比 entropy 更敏感？
- `conflict_aware_error_score` 的 AURC / E-AURC 是否低于 `1 - MSP_final`？


In [ ]:
def corrupt_labels(labels: torch.Tensor, num_classes: int, noise_rate: float, seed: int = 0) -> torch.Tensor:
    rng = np.random.default_rng(seed)
    y = labels.detach().cpu().long().numpy().copy()
    n = len(y)
    m = int(round(noise_rate * n))
    if m <= 0:
        return torch.tensor(y, dtype=torch.long)

    idx = rng.choice(n, size=m, replace=False)
    for i in idx:
        old = int(y[i])
        new = int(rng.integers(0, num_classes - 1))
        if new >= old:
            new += 1
        y[i] = new
    return torch.tensor(y, dtype=torch.long)

def summarize_noise_run(df: pd.DataFrame, noise_rate: float) -> dict:
    scores = [
        "js_text_support",
        "one_minus_cos_text_support",
        "top1_disagree_text_support",
        "margin_conflict_text_support",
        "one_minus_msp_final",
        "entropy_final",
        "energy_final",
        "one_minus_msp_support",
        "entropy_support",
        "conflict_aware_error_score",
    ]
    det = detection_table(df, "error_final", scores)

    def get_metric(score, col):
        x = det.loc[det.score == score, col]
        return float(x.iloc[0]) if len(x) else np.nan

    return {
        "noise_rate": noise_rate,
        "text_acc": float(df["correct_text"].mean()),
        "support_acc": float(df["correct_support"].mean()),
        "final_acc": float(df["correct_final"].mean()),
        "mean_js": float(df["js_text_support"].mean()),
        "mean_entropy_final": float(df["entropy_final"].mean()),
        "mean_msp_support": float(df["msp_support"].mean()),
        "auroc_js": get_metric("js_text_support", "auroc"),
        "auroc_entropy_final": get_metric("entropy_final", "auroc"),
        "auroc_1_minus_msp_final": get_metric("one_minus_msp_final", "auroc"),
        "auroc_conflict_aware": get_metric("conflict_aware_error_score", "auroc"),
        "aurc_1_minus_msp_final": get_metric("one_minus_msp_final", "aurc"),
        "aurc_conflict_aware": get_metric("conflict_aware_error_score", "aurc"),
        "eaurc_1_minus_msp_final": get_metric("one_minus_msp_final", "eaurc"),
        "eaurc_conflict_aware": get_metric("conflict_aware_error_score", "eaurc"),
    }

if RUN_EXP2:
    support = get_support_features_from_trainer(trainer)
    query_pack = collect_query_features(trainer)
    n_classes = len(classnames)

    exp2_rows = []
    exp2_metric_tables = []

    for noise in LABEL_NOISE_RATES:
        noisy_labels = corrupt_labels(support["labels"], n_classes, noise, seed=SEED + int(noise * 1000))
        pack = build_text_support_frame(
            trainer,
            support_features=support["features"],
            support_labels=noisy_labels,
            query_pack=query_pack,
            save_prefix=f"exp2_noise_{int(noise*100):02d}",
        )
        df_noise = pack["df"]
        df_noise["noise_rate"] = noise

        row = summarize_noise_run(df_noise, noise)
        exp2_rows.append(row)

        scores = [
            "js_text_support",
            "one_minus_cos_text_support",
            "top1_disagree_text_support",
            "margin_conflict_text_support",
            "one_minus_msp_final",
            "entropy_final",
            "energy_final",
            "one_minus_msp_support",
            "entropy_support",
            "conflict_aware_error_score",
        ]
        det = detection_table(df_noise, "error_final", scores)
        det["noise_rate"] = noise
        exp2_metric_tables.append(det)

    exp2_summary = pd.DataFrame(exp2_rows)
    exp2_metrics_all = pd.concat(exp2_metric_tables, ignore_index=True)

    exp2_summary_path = OUT_DIR / "exp2_label_noise_summary.csv"
    exp2_metrics_path = OUT_DIR / "exp2_label_noise_detection_metrics.csv"
    exp2_summary.to_csv(exp2_summary_path, index=False)
    exp2_metrics_all.to_csv(exp2_metrics_path, index=False)

    print("saved:", exp2_summary_path)
    print("saved:", exp2_metrics_path)
    display(exp2_summary)
else:
    print("RUN_EXP2=False")


## Experiment 3: prompt mismatch 下，能否识别 text prior 不可靠？

完整 wrong/weak/domain-shift prompts 需要重新 encode prompt templates。为了最小可行验证，这里先用三种 text prior perturbation 近似：

1. `original`: 原始 text features
2. `weak_blend_mean`: 把 text prototype 往全局均值拉近，模拟弱 prompt / 低判别性 prompt
3. `coarse_alphabetic_groups`: 按类别名排序分组求均值，模拟粗粒度 prompt
4. `wrong_permutation`: 随机打乱 text prototypes，模拟 wrong prompt / wrong class prior

看点：

- text prior 变坏时，support evidence 是否能修正 final prediction？
- text-support conflict 是否能识别 text prior error？
- conflict 高时，confidence 是否应该降低？


In [ ]:
def make_prompt_mismatch_text_features(trainer, mode: str, seed: int = 0, n_groups: int | None = None) -> torch.Tensor:
    base = trainer.method.model.adapter.base_text_features.detach().cpu().float()
    C, D = base.shape
    rng = np.random.default_rng(seed)

    if mode == "original":
        return base.clone()

    if mode == "weak_blend_mean":
        mean = base.mean(dim=0, keepdim=True)
        out = F.normalize(0.35 * base + 0.65 * mean, dim=-1)
        return out

    if mode == "wrong_permutation":
        perm = rng.permutation(C)
        # Avoid identity positions when possible.
        if C > 1:
            for i in range(C):
                if perm[i] == i:
                    j = (i + 1) % C
                    perm[i], perm[j] = perm[j], perm[i]
        return base[torch.tensor(perm, dtype=torch.long)].clone()

    if mode == "coarse_alphabetic_groups":
        names = list(trainer.dm.dataset.classnames)
        if n_groups is None:
            n_groups = max(2, int(round(math.sqrt(C))))
        order = np.argsort([str(x).lower() for x in names])
        out = base.clone()
        groups = np.array_split(order, n_groups)
        for g in groups:
            if len(g) == 0:
                continue
            mean = base[torch.tensor(g, dtype=torch.long)].mean(dim=0, keepdim=True)
            out[torch.tensor(g, dtype=torch.long)] = mean.expand(len(g), -1)
        out = F.normalize(out, dim=-1)
        return out

    raise ValueError(f"Unknown mismatch mode: {mode}")

def summarize_prompt_run(df: pd.DataFrame, mode: str) -> dict:
    # text prior error means zero-shot/text wrong, regardless of final.
    scores_text_error = [
        "js_text_support",
        "one_minus_cos_text_support",
        "top1_disagree_text_support",
        "margin_conflict_text_support",
        "one_minus_msp_text",
        "entropy_text",
        "ignorance_score",
    ]
    det_text = detection_table(df, "error_text", scores_text_error)

    scores_final_error = [
        "js_text_support",
        "one_minus_cos_text_support",
        "margin_conflict_text_support",
        "one_minus_msp_final",
        "entropy_final",
        "conflict_aware_error_score",
    ]
    det_final = detection_table(df, "error_final", scores_final_error)

    def get(det, score, col):
        x = det.loc[det.score == score, col]
        return float(x.iloc[0]) if len(x) else np.nan

    text_wrong_support_correct = ((df["error_text"] == 1) & (df["correct_support"] == 1)).mean()
    text_wrong_final_correct = ((df["error_text"] == 1) & (df["correct_final"] == 1)).mean()

    return {
        "prompt_mode": mode,
        "text_acc": float(df["correct_text"].mean()),
        "support_acc": float(df["correct_support"].mean()),
        "final_acc": float(df["correct_final"].mean()),
        "mean_js": float(df["js_text_support"].mean()),
        "text_wrong_support_correct_rate_all": float(text_wrong_support_correct),
        "text_wrong_final_correct_rate_all": float(text_wrong_final_correct),
        "auroc_js_for_text_error": get(det_text, "js_text_support", "auroc"),
        "auroc_disagree_for_text_error": get(det_text, "top1_disagree_text_support", "auroc"),
        "auroc_entropy_text_for_text_error": get(det_text, "entropy_text", "auroc"),
        "auroc_js_for_final_error": get(det_final, "js_text_support", "auroc"),
        "auroc_conflict_aware_for_final_error": get(det_final, "conflict_aware_error_score", "auroc"),
        "aurc_conflict_aware_for_final_error": get(det_final, "conflict_aware_error_score", "aurc"),
    }

if RUN_EXP3:
    support = get_support_features_from_trainer(trainer)
    query_pack = collect_query_features(trainer)

    exp3_rows = []
    exp3_metric_tables = []

    for mode in PROMPT_MISMATCH_MODES:
        text_features = make_prompt_mismatch_text_features(trainer, mode, seed=SEED)
        pack = build_text_support_frame(
            trainer,
            support_features=support["features"],
            support_labels=support["labels"],
            text_features_override=text_features,
            query_pack=query_pack,
            save_prefix=f"exp3_prompt_{mode}",
        )
        df_mode = pack["df"]
        df_mode["prompt_mode"] = mode

        exp3_rows.append(summarize_prompt_run(df_mode, mode))

        det = detection_table(df_mode, "error_text", [
            "js_text_support",
            "one_minus_cos_text_support",
            "top1_disagree_text_support",
            "margin_conflict_text_support",
            "one_minus_msp_text",
            "entropy_text",
            "ignorance_score",
        ])
        det["prompt_mode"] = mode
        det["target_error"] = "text_error"
        exp3_metric_tables.append(det)

    exp3_summary = pd.DataFrame(exp3_rows)
    exp3_metrics_all = pd.concat(exp3_metric_tables, ignore_index=True)

    exp3_summary_path = OUT_DIR / "exp3_prompt_mismatch_summary.csv"
    exp3_metrics_path = OUT_DIR / "exp3_prompt_mismatch_detection_metrics.csv"
    exp3_summary.to_csv(exp3_summary_path, index=False)
    exp3_metrics_all.to_csv(exp3_metrics_path, index=False)

    print("saved:", exp3_summary_path)
    print("saved:", exp3_metrics_path)
    display(exp3_summary)
else:
    print("RUN_EXP3=False")


## Experiment 4: OOD / near-OOD query 下，ignorance 与 conflict 是否可分离？

建议配置：

- support: ImageNet few-shot support
- ID query: ImageNet val/test
- near-OOD query: ImageNet-V2 / R / A / Sketch

目标区分三种状态：

1. `ignorance`: support density low
2. `conflict`: support density high but text-support disagreement high
3. `dominance`: one source dominates another source

这个实验默认关闭，因为 ImageNet 系列数据较重。确认数据路径和 configs 可用后，将 `RUN_EXP4=True`。


In [ ]:
def assign_states(df: pd.DataFrame, density_low_q: float = DENSITY_LOW_Q, conflict_high_q: float = CONFLICT_HIGH_Q, dominance_gap: float = DOMINANCE_GAP):
    out = df.copy()
    density_thr = out["support_density_topk_mean"].quantile(density_low_q)
    conflict_thr = out["js_text_support"].quantile(conflict_high_q)

    out["state"] = "mixed"
    out.loc[out["support_density_topk_mean"] <= density_thr, "state"] = "ignorance"
    out.loc[(out["support_density_topk_mean"] > density_thr) & (out["js_text_support"] >= conflict_thr), "state"] = "conflict"

    text_dom = (out["msp_text"] - out["msp_support"] >= dominance_gap) & (out["js_text_support"] >= conflict_thr)
    supp_dom = (out["msp_support"] - out["msp_text"] >= dominance_gap) & (out["js_text_support"] >= conflict_thr)
    out.loc[text_dom, "state"] = "text_dominance"
    out.loc[supp_dom, "state"] = "support_dominance"
    return out

def build_query_trainer_for_dataset(base_method_alias: str, query_dataset: str, shots: int, seed: int, out_subdir: Path):
    # Uses the query dataset's dataloader/transform, but does not require its checkpoint.
    return build_sanity_trainer(
        method_alias=base_method_alias,
        dataset=query_dataset,
        shots=shots,
        seed=seed,
        output_dir=out_subdir,
        load_checkpoint=False,
        model_dir=None,
    )

def exp4_run_ood():
    support_out = OUT_DIR / "exp4_support_imagenet"
    support_out.mkdir(parents=True, exist_ok=True)

    support_trainer = build_sanity_trainer(
        method_alias=METHOD_ALIAS,
        dataset=EXP4_SUPPORT_DATASET,
        shots=EXP4_SUPPORT_SHOTS,
        seed=SEED,
        output_dir=support_out,
        load_checkpoint=False,
    )
    support = get_support_features_from_trainer(support_trainer)
    support_text_features = support_trainer.method.model.adapter.base_text_features.detach().cpu().float()
    support_names = list(support_trainer.dm.dataset.classnames)

    all_rows = []
    id_query = collect_query_features(support_trainer)
    id_pack = build_text_support_frame(
        support_trainer,
        support_features=support["features"],
        support_labels=support["labels"],
        text_features_override=support_text_features,
        query_pack=id_query,
        save_prefix="exp4_query_imagenet_id",
    )
    id_df = assign_states(id_pack["df"])
    id_df["query_dataset"] = EXP4_SUPPORT_DATASET
    id_df["ood"] = 0
    all_rows.append(id_df)

    for qds in EXP4_QUERY_DATASETS:
        q_out = OUT_DIR / f"exp4_query_{qds}"
        q_out.mkdir(parents=True, exist_ok=True)
        q_trainer = build_query_trainer_for_dataset(METHOD_ALIAS, qds, EXP4_SUPPORT_SHOTS, SEED, q_out)

        q_names = list(q_trainer.dm.dataset.classnames)
        if len(q_names) != len(support_names):
            print(f"[WARN] class count mismatch for {qds}: {len(q_names)} vs support {len(support_names)}. Results may be invalid.")

        q_pack = collect_query_features(q_trainer)
        # Important: use support_trainer text features and support labels/class order.
        pack = build_text_support_frame(
            support_trainer,
            support_features=support["features"],
            support_labels=support["labels"],
            text_features_override=support_text_features,
            query_pack=q_pack,
            save_prefix=f"exp4_query_{qds}",
        )
        df = assign_states(pack["df"])
        df["query_dataset"] = qds
        df["ood"] = 1
        all_rows.append(df)

    full = pd.concat(all_rows, ignore_index=True)

    ood_scores = [
        "ignorance_score",
        "js_text_support",
        "one_minus_cos_text_support",
        "top1_disagree_text_support",
        "entropy_final",
        "one_minus_msp_final",
    ]
    rows = []
    for s in ood_scores:
        rows.append({"score": s, "ood_auroc": safe_roc_auc(full["ood"].to_numpy(), full[s].to_numpy())})
    ood_metric = pd.DataFrame(rows).sort_values("ood_auroc", ascending=False)

    state_summary = (
        full.groupby(["query_dataset", "ood", "state"])
        .size()
        .reset_index(name="count")
        .sort_values(["query_dataset", "state"])
    )

    full_path = OUT_DIR / "exp4_ood_state_samples.csv"
    metric_path = OUT_DIR / "exp4_ood_detection_metrics.csv"
    state_path = OUT_DIR / "exp4_ood_state_summary.csv"
    full.to_csv(full_path, index=False)
    ood_metric.to_csv(metric_path, index=False)
    state_summary.to_csv(state_path, index=False)

    print("saved:", full_path)
    print("saved:", metric_path)
    print("saved:", state_path)
    return full, ood_metric, state_summary

if RUN_EXP4:
    exp4_full, exp4_ood_metric, exp4_state_summary = exp4_run_ood()
    display(exp4_ood_metric)
    display(exp4_state_summary)
else:
    print("RUN_EXP4=False. Set RUN_EXP4=True after ImageNet support/query datasets are ready.")


## Final decision summary

这个 cell 汇总 4 个 sanity experiments 的关键结果，并给出是否值得继续的初步判断。


In [ ]:
decision = {
    "method_alias": METHOD_ALIAS,
    "dataset": DATASET,
    "shots": SHOTS,
    "seed": SEED,
    "out_dir": str(OUT_DIR),
    "experiments": {},
    "recommendation": None,
    "reasons": [],
}

if RUN_EXP1 and "exp1_metrics" in globals():
    def mget(df, score, col):
        x = df.loc[df.score == score, col]
        return float(x.iloc[0]) if len(x) else np.nan

    exp1_info = {
        "final_acc": float(exp1_df["correct_final"].mean()),
        "text_acc": float(exp1_df["correct_text"].mean()),
        "support_acc": float(exp1_df["correct_support"].mean()),
        "auroc_js": mget(exp1_metrics, "js_text_support", "auroc"),
        "auroc_margin_conflict": mget(exp1_metrics, "margin_conflict_text_support", "auroc"),
        "auroc_entropy_final": mget(exp1_metrics, "entropy_final", "auroc"),
        "auroc_1_minus_msp_final": mget(exp1_metrics, "one_minus_msp_final", "auroc"),
    }
    if "exp1_kill" in globals():
        exp1_info["delta_auroc_confidence_plus_js"] = float(exp1_kill.loc[exp1_kill.model == "confidence_plus_js", "delta_auroc_vs_base"].iloc[0])
        exp1_info["delta_eaurc_confidence_plus_js"] = float(exp1_kill.loc[exp1_kill.model == "confidence_plus_js", "delta_eaurc_vs_base"].iloc[0])
    decision["experiments"]["exp1"] = exp1_info

if RUN_EXP2 and "exp2_summary" in globals():
    decision["experiments"]["exp2"] = {
        "rows": exp2_summary.to_dict(orient="records"),
        "max_noise_auroc_js": float(exp2_summary["auroc_js"].max()),
        "max_noise_auroc_entropy_final": float(exp2_summary["auroc_entropy_final"].max()),
    }

if RUN_EXP3 and "exp3_summary" in globals():
    decision["experiments"]["exp3"] = {
        "rows": exp3_summary.to_dict(orient="records"),
        "wrong_permutation": exp3_summary.loc[exp3_summary.prompt_mode == "wrong_permutation"].to_dict(orient="records"),
    }

if RUN_EXP4 and "exp4_ood_metric" in globals():
    decision["experiments"]["exp4"] = {
        "ood_metrics": exp4_ood_metric.to_dict(orient="records"),
        "state_summary": exp4_state_summary.to_dict(orient="records"),
    }

# Conservative automatic recommendation.
kill = False
reasons = []

if "exp1" in decision["experiments"]:
    e1 = decision["experiments"]["exp1"]
    js = e1.get("auroc_js", np.nan)
    ent = e1.get("auroc_entropy_final", np.nan)
    msp = e1.get("auroc_1_minus_msp_final", np.nan)
    delta = e1.get("delta_auroc_confidence_plus_js", np.nan)

    if np.isfinite(js) and np.isfinite(ent) and js <= ent:
        kill = True
        reasons.append("Exp1: JS conflict AUROC is not higher than final entropy AUROC.")
    if np.isfinite(js) and np.isfinite(msp) and js <= msp:
        kill = True
        reasons.append("Exp1: JS conflict AUROC is not higher than final MSP baseline.")
    if np.isfinite(delta) and delta < 0.01:
        kill = True
        reasons.append("Exp1: adding JS after confidence improves AUROC by < 0.01.")

if "exp2" in decision["experiments"]:
    e2 = pd.DataFrame(decision["experiments"]["exp2"]["rows"])
    if len(e2) > 0:
        # At high noise, JS should beat entropy or conflict-aware AURC should improve.
        high = e2.sort_values("noise_rate").tail(1).iloc[0]
        if high["auroc_js"] <= high["auroc_entropy_final"]:
            reasons.append("Exp2: at highest label noise, JS is not more sensitive than entropy.")
        if high["aurc_conflict_aware"] >= high["aurc_1_minus_msp_final"]:
            reasons.append("Exp2: conflict-aware selective risk proxy does not improve AURC at highest noise.")

if "exp3" in decision["experiments"]:
    e3 = pd.DataFrame(decision["experiments"]["exp3"]["rows"])
    wrong = e3[e3.prompt_mode == "wrong_permutation"]
    if len(wrong) > 0:
        row = wrong.iloc[0]
        if row["auroc_js_for_text_error"] < 0.80:
            reasons.append("Exp3: JS does not strongly identify wrong text prior under wrong_permutation.")

if "exp4" not in decision["experiments"] and RUN_EXP4 is False:
    reasons.append("Exp4 not run yet; cannot claim ignorance/conflict/dominance separation.")

decision["recommendation"] = "KILL_OR_PAUSE" if kill else "CONTINUE_WITH_MORE_DATASETS"
decision["reasons"] = reasons

decision_path = OUT_DIR / "minimal_sanity_decision.json"
with open(decision_path, "w", encoding="utf-8") as f:
    json.dump(decision, f, indent=2, ensure_ascii=False)

print(json.dumps(decision, indent=2, ensure_ascii=False))
print("saved:", decision_path)


## Output files

主要输出文件：

- `exp1_text_support_conflict_samples.csv`
- `exp1_text_support_conflict_probs.npz`
- `exp1_error_detection_metrics.csv`
- `exp1_kill_test_confidence_plus_js.csv`
- `exp2_label_noise_summary.csv`
- `exp2_label_noise_detection_metrics.csv`
- `exp3_prompt_mismatch_summary.csv`
- `exp3_prompt_mismatch_detection_metrics.csv`
- `exp4_ood_state_samples.csv`，仅当 `RUN_EXP4=True`
- `exp4_ood_detection_metrics.csv`，仅当 `RUN_EXP4=True`
- `minimal_sanity_decision.json`

最重要的判断：

1. Exp1: `JS` 是否强于 `MSP / entropy / energy`
2. Exp2: label noise 高时 `JS` 是否强于 entropy，conflict-aware selective risk 是否下降
3. Exp3: prompt mismatch 时 `JS` 是否能识别 text prior error
4. Exp4: ignorance / conflict / dominance 三种状态是否统计上可分
